# 🎬 CineMatch - Sistema de Recomendación

Este notebook implementa un sistema de recomendación híbrido basado en:

- Preferencias de género del usuario (content-based)
- Popularidad de las películas (rating medio)

Objetivo:
Generar recomendaciones personalizadas sin necesidad de historial del usuario.

### Bloque 1: Importación y carga de datos

In [81]:
import pandas as pd
import numpy as np

ratings = pd.read_csv("../../../data/processed/ratings_sample.csv")
movies = pd.read_csv("../../../data/processed/movies_clean.csv")

### Bloque 2: Pre-procesado

In [82]:
## Convertimos en lista los generos

import ast

movies["genres"] = movies["genres"].apply(ast.literal_eval)

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]


### Bloque 3: Importación baseline

In [83]:
import sys
sys.path.append("../")

from baseline import train_baseline


model_df = train_baseline(ratings, movies)

model_df.head()

,movieId,rating,title,genres
0,2973,4.425000,Crimes and Misdemeanors (1989),"[Comedy, Crime, Drama]"
1,1237,4.400000,"Seventh Seal, The (Sjunde inseglet, Det) (1957)",[Drama]
2,318,4.385220,"Shawshank Redemption, The (1994)","[Crime, Drama]"
3,858,4.381773,"Godfather, The (1972)","[Crime, Drama]"
4,202439,4.378788,Parasite (2019),"[Comedy, Drama]"


### Bloque 4: Simulación de usuario

In [84]:
user_preferences = {
    "Action": 1.0,
    "Comedy": 0.9,
    "Sci-Fi": 0.9
}

### Bloque 5: Función Scoring

In [85]:
def compute_genre_score(movie_genres, user_prefs):
    scores = [user_prefs.get(g, 0) for g in movie_genres]
    return sum(scores) / len(movie_genres)

### Bloque 6: Aplicar Scoring

In [86]:
df = model_df.copy()

df["genre_score"] = df["genres"].apply(
    lambda g: compute_genre_score(g, user_preferences)
)

df.head()

,movieId,rating,title,genres,genre_score
0,2973,4.425000,Crimes and Misdemeanors (1989),"[Comedy, Crime, Drama]",0.30
1,1237,4.400000,"Seventh Seal, The (Sjunde inseglet, Det) (1957)",[Drama],0.00
2,318,4.385220,"Shawshank Redemption, The (1994)","[Crime, Drama]",0.00
3,858,4.381773,"Godfather, The (1972)","[Crime, Drama]",0.00
4,202439,4.378788,Parasite (2019),"[Comedy, Drama]",0.45


### Bloque 7: Normalizacion de rating

In [87]:
df["rating_norm"] = df["rating"] / 5.0

### Bloque 8: Score final

In [88]:
df["final_score"] = (
    0.7 * df["genre_score"] +
    0.3 * df["rating_norm"]
)

In [89]:
recommendations = df.sort_values(
    by="final_score",
    ascending=False
)

recommendations.head(10)[
    ["title", "rating", "genres", "genre_score", "final_score"]
]

,title,rating,genres,genre_score,final_score
170,Terminator 2: Judgment Day (1991),3.992611,"[Action, Sci-Fi]",0.950000,0.904557
367,Kung Fu Hustle (Gong fu) (2004),3.812500,"[Action, Comedy]",0.950000,0.893750
197,Deadpool 2 (2018),3.964286,"[Action, Comedy, Sci-Fi]",0.933333,0.891190
477,Logan (2017),3.726190,"[Action, Sci-Fi]",0.950000,0.888571
556,Red (2010),3.666667,"[Action, Comedy]",0.950000,0.885000
417,Ghostbusters (a.k.a. Ghost Busters) (1984),3.771552,"[Action, Comedy, Sci-Fi]",0.933333,0.879626
707,Kick-Ass (2010),3.545455,"[Action, Comedy]",0.950000,0.877727
86,National Lampoon's Vacation (1983),4.100000,[Comedy],0.900000,0.876000
128,Monty Python's Life of Brian (1979),4.041176,[Comedy],0.900000,0.872471
151,Sleeper (1973),4.019231,"[Comedy, Sci-Fi]",0.900000,0.871154
